# Let PyTorch find the gradients

Room310 · Deep learning foundations

## Goal

Rebuild the line learner with PyTorch and explain why gradient clearing and no_grad matter.

## Setup

Run this notebook from top to bottom. It is self-contained and uses only synthetic teaching data. No credentials or dataset downloads are needed. Save a copy before editing.

Use a CPU Python environment with PyTorch installed. In Colab, connect to the default CPU runtime. If `import torch` fails, run `%pip install torch` in a separate cell and restart the kernel if asked. For local installation, follow https://pytorch.org/get-started/locally/.

Examples use a fixed seed where relevant; exact floating-point results can vary by environment.

## Steps

### 1. Autograd records the calculation

**Autograd** is PyTorch's automatic differentiation system. Mark a parameter with `requires_grad=True`, then calculate a loss using it. PyTorch records how those operations depend on the parameter. Calling `loss.backward()` works backward through that calculation and stores derivatives in `.grad`.

This is **backpropagation**: applying the chain rule backward through a computation graph. It calculates gradients; it does not update parameters by itself.

In [1]:
import torch

weight = torch.tensor(0.0, requires_grad=True)
bias = torch.tensor(1.0, requires_grad=True)
prediction = weight * 3.0 + bias
loss = (prediction - 7.0) ** 2
loss.backward()

print("Weight gradient:", weight.grad.item())
print("Bias gradient:", bias.grad.item())

Weight gradient: -36.0
Bias gradient: -12.0


**Check your result:** The weight gradient is -36.0 and the bias gradient is -12.0. The weight gradient matches our numerical slope from lesson 2.

### 2. Predict → measure → backpropagate → update

Now train on the same five input/target pairs. We explicitly start new parameters at zero, so this cell does not reuse the previous example's gradients.

PyTorch **accumulates gradients** by default. Clear them before each fresh backward pass. Use `torch.no_grad()` during the manual update so changing a parameter is not recorded as part of the next prediction graph. `.item()` turns a one-element tensor into a Python number for printing.

Every iteration builds a fresh forward calculation. You normally do not call backward twice on the same loss object; calculate a new loss on the next iteration.

In [2]:
X = torch.tensor([[-2.0], [-1.0], [0.0], [1.0], [2.0]])
y = 2 * X + 1
weight = torch.tensor(0.0, requires_grad=True)
bias = torch.tensor(0.0, requires_grad=True)
learning_rate = 0.1

for epoch in range(60):
    weight.grad = None
    bias.grad = None
    predictions = weight * X + bias
    assert predictions.shape == y.shape
    loss = ((predictions - y) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        weight -= learning_rate * weight.grad
        bias -= learning_rate * bias.grad
    if epoch % 20 == 0:
        print(f"Epoch {epoch:2d} | loss before update: {loss.item():.6f}")

print(f"Learned weight: {weight.item():.3f}, bias: {bias.item():.3f}")

Epoch  0 | loss before update: 9.000000
Epoch 20 | loss before update: 0.000133
Epoch 40 | loss before update: 0.000000
Learned weight: 2.000, bias: 1.000


**Check your result:** The parameters should again approach weight 2.000 and bias 1.000. Compare this loop with lesson 2: PyTorch replaced the derivative formulas, not the learning process.

### 3. Use the model without training it

**Inference** means using learned parameters to make predictions. We do not need to record gradients while doing that. `torch.no_grad()` reduces unnecessary tracking; it does not erase the learned numbers.

In the next lesson, an optimizer will handle the update and a neural-network module will hold the parameters. Keep this manual version nearby as a map of what those tools do.

In [3]:
with torch.no_grad():
    new_inputs = torch.tensor([[3.0], [4.0]])
    new_predictions = weight * new_inputs + bias
print("New predictions:", new_predictions.flatten().tolist())

New predictions: [6.999998569488525, 8.999998092651367]


**Check your result:** Expect values close to 7 and 9. Extrapolation works here because we invented a perfect line; real datasets need more careful evaluation.

## Checks

Autograd calculates gradients. Your training loop still decides when to clear them, how to update parameters, and when to stop.

Compare your output with each check above. Explain unexpected results before moving on.

## Next Steps

### Practice & explain

### Explain four lines

In your own words, explain grad = None, loss.backward(), torch.no_grad(), and weight -= learning_rate * weight.grad. Identify which line computes a gradient and which changes a parameter.

<details><summary>Need a hint?</summary>

backward computes; subtraction updates. The other two control gradient bookkeeping.

</details>

In [4]:
# Your experiment or explanation goes here.


### Prove gradients accumulate

In a new cell create p = torch.tensor(2.0, requires_grad=True). Run (p ** 2).backward() twice, printing p.grad after each call. Reset p.grad to None and try once more. Explain 4, 8, and 4.

<details><summary>Need a hint?</summary>

Each (p ** 2) expression creates a fresh graph. Without clearing, the newly computed derivative is added to the old gradient.

</details>

In [5]:
# Your experiment or explanation goes here.


### References

- [PyTorch · automatic differentiation](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html)
- [PyTorch · optimizing model parameters](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)